In [1]:
# sr_benchmark_run_ceql.py

from __future__ import annotations

import csv
import time
from pathlib import Path

import h5py
import numpy as np
import sympy as sp
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from config.benchmark_config import DataCFG, CEQL_TRAIN, CEQL
from src.ComplexEQL import ComplexEQL
from src.utils import set_seed, train


class RelativeMSELoss(nn.Module):
    def __init__(self, eps: float = 1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, yhat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        r = yhat - y
        denom = y.abs() + self.eps
        return ((r / denom) ** 2).mean()


class MSEOrRelativeMSELoss(nn.Module):
    def __init__(self, pivot: float = 1.0):
        super().__init__()
        self.pivot = pivot

    def forward(self, yhat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        r = yhat - y
        denom = torch.maximum(y.abs(), y.new_tensor(self.pivot))
        return ((r / denom) ** 2).mean()


def _load_one_group(f: h5py.File, gname: str):
    g = f[gname]
    raw = g["sympy_str"][()]
    true_expr_str = raw.decode("utf-8") if isinstance(raw, (bytes, bytearray)) else str(raw)

    Xtr = g["train"]["X"][...].astype(np.float32, copy=False)
    ytr = g["train"]["y"][...].astype(np.float32, copy=False).reshape(-1)

    Xti = g["test_interp"]["X"][...].astype(np.float32, copy=False)
    yti = g["test_interp"]["y"][...].astype(np.float32, copy=False).reshape(-1)

    Xte = g["test_extrap"]["X"][...].astype(np.float32, copy=False)
    yte = g["test_extrap"]["y"][...].astype(np.float32, copy=False).reshape(-1)

    return true_expr_str, Xtr, ytr, Xti, yti, Xte, yte


def _mse(yhat: np.ndarray, y: np.ndarray) -> float:
    yhat = np.asarray(yhat, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    return float(np.mean((yhat - y) ** 2))


def main():
    cfg = DataCFG()

    out_csv = Path(getattr(CEQL_TRAIN, "results_path", "reports/sr_benchmark_ceql.csv"))
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    n_runs = int(getattr(CEQL_TRAIN, "n_runs", 1))
    base_seed = int(getattr(CEQL_TRAIN, "base_seed", 0))

    device = torch.device(getattr(CEQL_TRAIN, "device", "cpu"))

    with h5py.File(cfg.h5_path, "r") as f, out_csv.open("w", newline="") as out:
        w = csv.writer(out)
        w.writerow(
            [
                "group",
                "run",
                "seed",
                "n_train",
                "n_features",
                "train_mse",
                "test_interp_mse",
                "test_extrap_mse",
                "duration_s",
                "true_expr",
                "found_expr",
            ]
        )

        groups = sorted(list(f.keys()))
        print(groups)

        for gname in groups:
            if gname in [
                'expr_000_lin_uni', 
                'expr_001_lin_bi', 
                'expr_002_poly2_uni', 
                'expr_003_poly2_bi', 
                # 'expr_004_log_poly2_sign2', 
                # 'expr_005_sqrt_poly2_sign2', 
                'expr_006_rat_linlin_uni_pole_train', 
                'expr_007_rat_linlin_bi_pole_train', 
                'expr_008_rat_poly2poly2_uni_one_pole_train', 
                'expr_009_rat_poly2poly2_uni_two_poles_train',
                ]:
                continue
            true_expr_str, Xtr, ytr, Xti, yti, Xte, yte = _load_one_group(f, gname)

            n_features = int(Xtr.shape[1])
            CEQL.n_input_fields = n_features

            Xtr_t = torch.tensor(Xtr, device=device)
            ytr_t = torch.tensor(ytr.reshape(-1, 1), device=device)

            dl = DataLoader(
                TensorDataset(Xtr_t, ytr_t),
                batch_size=int(getattr(CEQL_TRAIN, "train_batch_size", 2**14)),
                shuffle=True,
                drop_last=False,
            )

            for run_i in range(n_runs):
                seed = base_seed + run_i
                set_seed(seed)

                model = ComplexEQL(CEQL).to(device)
                loss_fn = MSEOrRelativeMSELoss(pivot=10.0)
                # loss_fn = torch.nn.MSELoss()

                opt = torch.optim.Adam(model.parameters(),
                                       lr=float(getattr(CEQL_TRAIN, "lr", 1e-3)),
                                       betas=(0.9, 0.999))

                sched = None
                if getattr(CEQL_TRAIN, "scheduler", None) == "ReduceLROnPlateau":
                    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                        opt, **getattr(CEQL_TRAIN, "schedulerparams", {})
                    )

                t0 = time.perf_counter()
                model, _ = train(
                    model=model,
                    dataloader=dl,
                    optimizer=opt,
                    loss_fn=loss_fn,
                    cfg=CEQL_TRAIN,
                    device=device,
                    scheduler=sched,
                    on_print=None,
                )
                dur = time.perf_counter() - t0

                model.eval()
                with torch.no_grad():
                    yhat_tr = model(torch.tensor(Xtr, device=device)).real.squeeze(-1).cpu().numpy()
                    yhat_ti = model(torch.tensor(Xti, device=device)).real.squeeze(-1).cpu().numpy()
                    yhat_te = model(torch.tensor(Xte, device=device)).real.squeeze(-1).cpu().numpy()

                train_mse = _mse(yhat_tr, ytr)
                test_interp_mse = _mse(yhat_ti, yti)
                test_extrap_mse = _mse(yhat_te, yte)

                found_expr_str = ""
                try:
                    if n_features == 1:
                        syms = [sp.Symbol("x1")]
                    else:
                        syms = [sp.Symbol(f"x{i+1}") for i in range(n_features)]

                    found = model.get_symbolic_expression(syms, rounding_decimals=5, use_imag=False)
                    found_expr_str = "" if found is None else str(found)
                except Exception:
                    found_expr_str = ""

                w.writerow(
                    [
                        gname,
                        run_i,
                        seed,
                        int(Xtr.shape[0]),
                        n_features,
                        train_mse,
                        test_interp_mse,
                        test_extrap_mse,
                        dur,
                        true_expr_str,
                        found_expr_str,
                    ]
                )
                out.flush()

                print(
                    f"[{gname}] run={run_i} seed={seed} "
                    f"train={train_mse:.3e} interp={test_interp_mse:.3e} extrap={test_extrap_mse:.3e} "
                    f"dur={dur:.2f}s"
                )
                if found_expr_str:
                    print("Found:", found_expr_str)

    print(f"\nSaved: {out_csv}")


if __name__ == "__main__":
    main()


['expr_000_lin_uni', 'expr_001_lin_bi', 'expr_002_poly2_uni', 'expr_003_poly2_bi', 'expr_004_log_poly2_sign2', 'expr_005_sqrt_poly2_sign2', 'expr_006_rat_linlin_uni_pole_train', 'expr_007_rat_linlin_bi_pole_train', 'expr_008_rat_poly2poly2_uni_one_pole_train', 'expr_009_rat_poly2poly2_uni_two_poles_train']
Random seed set as 0
[PHASE1 | Epoch 1] lr=1.00e-03, total=4.4006e-02, data=4.4006e-02, sparsity_reg=1.7399e-08, imag_w=5.5829e-08, theta=2.3126e-11, valid=1.000, active_edges=79
[PHASE1 | Epoch 1000] lr=1.00e-03, total=2.8373e-03, data=2.8372e-03, sparsity_reg=1.8671e-08, imag_w=5.3502e-08, theta=4.2506e-11, valid=1.000, active_edges=79
[PHASE1 | Epoch 2000] lr=1.00e-03, total=8.6418e-04, data=8.6411e-04, sparsity_reg=1.9087e-08, imag_w=5.3052e-08, theta=9.4368e-11, valid=1.000, active_edges=79
[PHASE1 | Epoch 3000] lr=1.00e-03, total=5.6261e-04, data=5.6254e-04, sparsity_reg=1.9221e-08, imag_w=5.2723e-08, theta=1.4678e-10, valid=1.000, active_edges=79
[PHASE1 | Epoch 4000] lr=1.00e